# What CLIP actually is

CLIP (Contrastive Language-Image Pre-training) is two separate encoders trained together:

An image encoder (a ViT or ResNet) that turns an image into a vector
A text encoder (a Transformer) that turns text into a vector

They're trained on ~400M (image, caption) pairs scraped from the internet, with a contrastive loss: for each batch, the correct image-caption pairs should have high cosine similarity, and every mismatched pair in that batch should have low similarity. Over millions of examples, this forces both encoders to project into a shared embedding space — even though the two encoders have completely different architectures internally, their outputs live in the same coordinate system.

That's the property you actually care about for MedRAG: once you have an image embedding and a text embedding in the same space, cosine similarity between them is meaningful. A text query like "chest X-ray showing pneumonia" can be compared directly against a database of image vectors — no separate "caption this image" step needed.

One thing to be upfront about, since it matters for a medical RAG system: CLIP was trained on general internet images, not medical imaging. It'll do fine on your WHO images (mostly diagrams, charts, algorithm flowcharts, not radiology scans), but if you ever added actual clinical images (X-rays, histology slides), general CLIP would be a weak choice — there are medical-domain variants (BiomedCLIP, PMC-CLIP) for that. Worth a one-line note in your report; not a blocker for what you're doing now.

# Load and Inspect WHO Image Metadata

In [1]:
import sys, os
from medrag.ingestion.storage import load_who_images

backend_path = os.path.abspath(os.path.join("..", "backend"))
sys.path.insert(0, backend_path)


WHO_TOPICS = [
    "tuberculosis", "hypertension", "diabetes", "obesity", "asthma", "copd",
    "coronary artery disease", "heart failure", "stroke", "hyperlipidemia",
    "pneumonia", "covid-19", "malaria", "hiv aids", "hepatitis b", "hepatitis c",
    "dengue fever", "typhoid", "depression", "anxiety disorder", "epilepsy",
    "malnutrition", "anemia in pregnancy", "breast cancer",
]

all_images = []
for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir="../data/images/who")
    all_images.extend(images)
    print(f"{topic}: {len(images)} images")

print()
print(f"Total image entries across all topic files: {len(all_images)}")

tuberculosis: 0 images
hypertension: 9 images
diabetes: 2 images
obesity: 0 images
asthma: 3 images
copd: 3 images
coronary artery disease: 3 images
heart failure: 3 images
stroke: 3 images
hyperlipidemia: 3 images
pneumonia: 4 images
covid-19: 4 images
malaria: 4 images
hiv aids: 0 images
hepatitis b: 8 images
hepatitis c: 4 images
dengue fever: 21 images
typhoid: 2 images
depression: 6 images
anxiety disorder: 6 images
epilepsy: 6 images
malnutrition: 4 images
anemia in pregnancy: 2 images
breast cancer: 0 images

Total image entries across all topic files: 100


# Deduplicate Images by Content Hash, Grouped by Topics

In [2]:
import hashlib
from pathlib import Path
from collections import defaultdict

def compute_file_hash(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


IMAGE_DIR = "../data/images/who"

hash_to_topics = defaultdict(list)
hash_to_image_record = {}

for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir=IMAGE_DIR)
    for img in images:
        filepath = Path(IMAGE_DIR) / img.filename
        if not filepath.exists():
            continue
        file_hash = compute_file_hash(filepath)
        hash_to_topics[file_hash].append(topic)
        if file_hash not in hash_to_image_record:
            hash_to_image_record[file_hash] = img

print(f"Total image entries: {sum(len(load_who_images(t, output_dir=IMAGE_DIR)) for t in WHO_TOPICS)}")
print(f"Unique images (by content hash): {len(hash_to_image_record)}")
print()
print("Examples of shared images:")
for h, topics in list(hash_to_topics.items())[:5]:
    if len(topics) > 1:
        print(f"  {hash_to_image_record[h].filename}: {topics}")

Total image entries: 100
Unique images (by content hash): 75

Examples of shared images:


# List All Shared Images

In [3]:
shared_count = 0
for h, topics in hash_to_topics.items():
    if len(topics) > 1:
        shared_count += 1
        print(f"  {hash_to_image_record[h].filename}: {topics}")

print()
print(f"Number of images shared across topics: {shared_count}")

# Also check for any images that failed to load (missing files), which could explain the off-by-one
missing_count = 0
for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir=IMAGE_DIR)
    for img in images:
        filepath = Path(IMAGE_DIR) / img.filename
        if not filepath.exists():
            missing_count += 1
            print(f"  MISSING FILE: {img.filename} (referenced under topic '{topic}')")

print(f"Missing files: {missing_count}")

  asthma_page9_img0.png: ['asthma', 'copd']
  asthma_page10_img1.png: ['asthma', 'copd']
  asthma_page78_img2.png: ['asthma', 'copd']
  coronary_artery_disease_page12_img0.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page13_img1.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page17_img2.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  pneumonia_page0_img0.png: ['pneumonia', 'malnutrition']
  depression_page19_img0.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page55_img1.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page70_img2.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page18_img3.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page21_img4.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page22_img5.png: ['depression', 'anxiety disorder', 'epilepsy'

# Exclude Cover-Page-Position Images From Cross-Topic Merging

In [4]:
import re

COVER_PAGE_PATTERN = re.compile(r"_page0_img0\.png$")

hash_to_topics = defaultdict(list)
hash_to_image_record = {}

for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir=IMAGE_DIR)
    for img in images:
        filepath = Path(IMAGE_DIR) / img.filename
        if not filepath.exists():
            continue

        if COVER_PAGE_PATTERN.search(img.filename):
            # Cover-page-position images are never merged across topics,
            # even if their content hash matches another document's cover
            # graphic (a shared PDF template banner, not real content) -
            # use a per-topic-unique key instead of the content hash
            key = f"{img.filename}_{topic}"
        else:
            key = compute_file_hash(filepath)

        hash_to_topics[key].append(topic)
        if key not in hash_to_image_record:
            hash_to_image_record[key] = img

print(f"Unique images (with cover-page exclusion rule): {len(hash_to_image_record)}")
print()
print("Shared images (excluding cover-page graphics):")
for h, topics in hash_to_topics.items():
    if len(topics) > 1:
        print(f"  {hash_to_image_record[h].filename}: {topics}")

Unique images (with cover-page exclusion rule): 76

Shared images (excluding cover-page graphics):
  asthma_page9_img0.png: ['asthma', 'copd']
  asthma_page10_img1.png: ['asthma', 'copd']
  asthma_page78_img2.png: ['asthma', 'copd']
  coronary_artery_disease_page12_img0.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page13_img1.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page17_img2.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  depression_page19_img0.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page55_img1.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page70_img2.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page18_img3.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page21_img4.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page22_img5.png: ['de